In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb

In [3]:
# ── 1. LOAD DATA ──────────────────────────────────────────────
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')


In [4]:
# ── 2. FEATURE ENGINEERING ───────────────────────────────────
# WHY: Raw features alone miss relationships between bands.
# Astronomers use "color indices" (band differences) as key classifiers.
def add_features(df):
    # Color indices — already in your v3 ✅
    df['u_minus_g'] = df['u'] - df['g']
    df['g_minus_r'] = df['g'] - df['r']
    df['r_minus_i'] = df['r'] - df['i']
    df['i_minus_z'] = df['i'] - df['z']
    df['total_mag']  = df['u'] + df['g'] + df['r'] + df['i'] + df['z']
    
    # NEW: Redshift is the single most powerful feature for QSOs.
    # Squaring it amplifies extreme values (QSOs have very high redshift).
    df['redshift_sq'] = df['redshift'] ** 2
    df['redshift_log'] = np.log1p(df['redshift'].clip(lower=0))  # log1p avoids log(0)

    # NEW: Cross-band ratios — capture spectral shape
    df['u_over_z']  = df['u'] / (df['z'] + 1e-6)
    df['g_over_r']  = df['g'] / (df['r'] + 1e-6)
    
    # NEW: Magnitude spread — how "wide" is the spectral profile?
    df['mag_range'] = df['u'] - df['z']
    df['mag_std']   = df[['u','g','r','i','z']].std(axis=1)
    
    return df

train_df = add_features(train_df)
test_df  = add_features(test_df)

In [5]:
# ── 3. ENCODE TARGET & CATEGORICALS ──────────────────────────
# WHY: Models need numbers, not strings.
le_target = LabelEncoder()
y = le_target.fit_transform(train_df['class'])  # GALAXY=0, QSO=1, STAR=2

cat_cols = ['spectral_type', 'galaxy_population']
for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col]  = le.transform(test_df[col])

drop_cols = ['id', 'class']
X = train_df.drop(drop_cols, axis=1)
X_test = test_df.drop(['id'], axis=1)

In [6]:
# ── 4. YOUR BEST PARAMS FROM OPTUNA ──────────────────────────
# Using your Trial 37 best result — no need to re-run Optuna
lgb_params = {
    'objective': 'multiclass',   # ← FIX: was missing in your final model!
    'num_class': 3,              # ← FIX: was missing!
    'metric': 'multi_logloss',
    'verbose': -1,
    'n_jobs': -1,
    'learning_rate': 0.10468,
    'num_leaves': 114,
    'max_depth': 8,
    'reg_alpha': 0.02327,
    'reg_lambda': 0.55586,
    'min_child_samples': 72,
    'subsample': 0.72157,
    'colsample_bytree': 0.79810,
    'n_estimators': 2000,        # Higher — early stopping will find the right point
}

xgb_params = {
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'verbosity': 0,
    'n_jobs': -1,
    'learning_rate': 0.05,
    'max_depth': 7,
    'n_estimators': 2000,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
}



In [7]:
# ── 5. OOF TRAINING LOOP ─────────────────────────────────────
# WHY: Instead of one model, train 5 models (one per fold).
# Average their test predictions → much more stable than a single model.
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Arrays to accumulate predictions
oof_lgb  = np.zeros((len(X), 3))       # OOF probabilities for validation
oof_xgb  = np.zeros((len(X), 3))
test_lgb = np.zeros((len(X_test), 3))  # Test probabilities (averaged across folds)
test_xgb = np.zeros((len(X_test), 3))

lgb_scores, xgb_scores = [], []

print("Starting OOF Training...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"── Fold {fold+1}/{N_SPLITS} ──")
    
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    
    # ── LightGBM ──
    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100, verbose=False),  # stop if no improvement in 100 rounds
                   lgb.log_evaluation(500)]
    )
    oof_lgb[val_idx]  = lgb_model.predict_proba(X_val)
    test_lgb         += lgb_model.predict_proba(X_test) / N_SPLITS  # accumulate average
    lgb_score = balanced_accuracy_score(y_val, oof_lgb[val_idx].argmax(axis=1))
    lgb_scores.append(lgb_score)
    print(f"  LGB Balanced Acc: {lgb_score:.4f}")
    
    # ── XGBoost ──
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    oof_xgb[val_idx]  = xgb_model.predict_proba(X_val)
    test_xgb         += xgb_model.predict_proba(X_test) / N_SPLITS
    xgb_score = balanced_accuracy_score(y_val, oof_xgb[val_idx].argmax(axis=1))
    xgb_scores.append(xgb_score)
    print(f"  XGB Balanced Acc: {xgb_score:.4f}\n")

print(f"LGB OOF Mean: {np.mean(lgb_scores):.4f}")
print(f"XGB OOF Mean: {np.mean(xgb_scores):.4f}")

Starting OOF Training...

── Fold 1/5 ──
  LGB Balanced Acc: 0.9557


TypeError: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'